# DiabCare AI — LightGBM Model Training & Evaluation

This notebook covers the training, validation, hyperparameter tuning, and comprehensive evaluation of the **LightGBM Classifier** for predicting 30-day hospital readmissions in diabetic patients.

### Metrics Evaluated:
- **Accuracy**
- **F1 Score**
- **Recall (Sensitivity)**
- **Precision**
- **ROC-AUC Score**
- **PR-AUC (Precision-Recall AUC)**
- **Confusion Matrix & Classification Report**

In [1]:
import sys
import os
import joblib
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

# Ensure project root is in sys.path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(""))))
from Src.Preprocessing import load_data, build_preprocessor

In [2]:
# Load cleaned dataset & perform Stratified Train-Test split
X, y = load_data("../DATA/CleanedDiabetic_data.csv")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Features shape: {X.shape} | Target distribution: {dict(y.value_counts())}")
print(f"Train set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")

Loading dataset from DATA/CleanedDiabetic_data.csv...
Features shape: (101766, 44) | Target distribution: {0: 90409, 1: 11357}
Train set: 81412 samples | Test set: 20354 samples


In [3]:
# Load saved model pipeline or train LightGBM Classifier
model_path = "../DATA/lgbm_pipeline.joblib"
if os.path.exists(model_path):
    pipeline = joblib.load(model_path)
    print(f"Loading pre-trained LightGBM Pipeline from {model_path}...")
else:
    print("Training new LightGBM Classifier pipeline...")
    pipeline = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("model", LGBMClassifier(
            class_weight="balanced",
            learning_rate=0.05,
            n_estimators=200,
            num_leaves=31,
            random_state=42,
            verbose=-1
        ))
    ])
    pipeline.fit(X_train, y_train)

print("Model successfully loaded:")
print(pipeline)

Loading pre-trained LightGBM Pipeline from ../DATA/lgbm_pipeline.joblib...
Model successfully loaded:
Pipeline(steps=[('preprocessor', ColumnTransformer(...)), ('model', LGBMClassifier(class_weight='balanced', learning_rate=0.05, n_estimators=200, num_leaves=31, random_state=42, verbose=-1))])


In [4]:
# Make predictions on the holdout Test set
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=" * 50)
print("     LIGHTGBM MODEL EVALUATION RESULTS")
print("=" * 50)
print(f"Accuracy         : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"ROC-AUC          : {roc_auc:.4f}")
print(f"PR-AUC           : {pr_auc:.4f}")
print(f"Precision        : {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall           : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1 Score         : {f1:.4f}")
print("=" * 50)

     LIGHTGBM MODEL EVALUATION RESULTS
Accuracy         : 0.6620 (66.20%)
ROC-AUC          : 0.6859
PR-AUC           : 0.2373
Precision        : 0.1858 (18.58%)
Recall           : 0.6002 (60.02%)
F1 Score         : 0.2838


In [5]:
# Detailed Confusion Matrix and Classification Report
cm = confusion_matrix(y_test, y_pred)
cr = classification_report(y_test, y_pred)

print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(cr)

Confusion Matrix:
 [[12111  5972]
 [  908  1363]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.67      0.78     18083
           1       0.19      0.60      0.28      2271

    accuracy                           0.66     20354
   macro avg       0.56      0.63      0.53     20354
weighted avg       0.85      0.66      0.72     20354
